# Adaptive Cyber Threat Detection for Rail SCADA Systems
## A Hybrid Machine Learning and Statistical Approach

**REPS UTC final report notebook.** Benjamin Gyimah Boateng and Nii Attoh-Okine, University of Maryland.

This notebook produces every table and figure used in the final report from one run.

**Detectors compared**

| Name | What it is |
|---|---|
| IF | Isolation Forest on the standardized features (baseline) |
| PCA | PCA reconstruction error (classical statistical baseline) |
| VAE | Variational Autoencoder reconstruction error (standalone VAE) |
| VAE+RM+MAD | Hybrid. VAE error, then a rolling median filter, then a MAD threshold |

**Ablation (other statistical models placed after the VAE)**: EWMA, CUSUM, Mahalanobis distance, robust z score.
Two detectors without a VAE are also included to show what the VAE contributes.

**How to run**

1. Put `WRM_F.csv` in your Google Drive (or next to this notebook).
2. Set `DATA_PATH` in Section 0 if needed.
3. Runtime > Run all. A CPU runtime is enough. The full run takes roughly 15 to 25 minutes.

All outputs are written to `OUT_DIR` (`tables/` as CSV, `figures/` as 300 dpi PNG).

## 0. Setup and configuration

In [ ]:
import os, json, warnings, time
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

# ---- paths -------------------------------------------------------------
DATA_PATH = '/content/drive/My Drive/WRM_F.csv'      # change if your file is elsewhere
OUT_DIR   = '/content/drive/My Drive/FDI_UTC_Results'

try:                                                 # Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:                                    # local run
    if not os.path.exists(DATA_PATH):
        DATA_PATH, OUT_DIR = 'WRM_F.csv', 'FDI_UTC_Results'

TAB_DIR, FIG_DIR = os.path.join(OUT_DIR, 'tables'), os.path.join(OUT_DIR, 'figures')
os.makedirs(TAB_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print('Data :', DATA_PATH); print('Out  :', OUT_DIR)

In [ ]:
# ---- experiment configuration (every setting of the study is here) -------
CONFIG = dict(
    features=['LProf62', 'RProf62', 'LAlign62', 'RAlign62', 'Gage', 'Warp62',
              'BFI_L_Value', 'BFI_C_Value', 'BFI_R_Value', 'Crosslevel',
              'TotalLatForce', 'Left Cant', 'Right Cant', 'Speed'],
    targets=['Gage', 'Warp62', 'Crosslevel'],          # attacked features
    window_len=46,                                     # records per attack window
    windows={0.02: 8, 0.05: 20, 0.10: 40},             # 368, 920, 1840 attacked records
    noise_std=3.0,                                     # noise attack, in feature standard deviations
    # VAE
    latent_dim=7, hidden=(64, 32, 16), epochs=100, batch_size=64, patience=10,
    # hybrid
    rm_window=11,                                      # rolling median window (records)
    k_mad=3.0,                                         # conventional MAD multiplier
    target_far=0.01,                                   # false alarm rate used to calibrate the MAD multiplier
    # other statistical models (ablation)
    ewma_lambda=0.2, cusum_k=1.0,
    # thresholds
    percentile=95, n_folds=5,
    # repetitions. These seeds were not used when the design was chosen.
    seeds=list(range(100, 110)),
    holdout_seeds=list(range(100, 105)), holdout_train_frac=0.6,
    rm_window_grid=[3, 5, 11, 21, 31], k_grid=[3, 5, 8, 12],
)
MAIN_MODELS = ['IF', 'PCA', 'VAE', 'VAE+RM+MAD']
ABLATION    = ['VAE', 'VAE+RM+MAD', 'VAE+EWMA', 'VAE+CUSUM', 'VAE+Mahalanobis', 'VAE+robust z',
               'Robust z (no VAE)', 'RM robust z (no VAE)']
json.dump(CONFIG, open(os.path.join(OUT_DIR, 'config.json'), 'w'), indent=1)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.covariance import MinCovDet
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, average_precision_score, roc_curve)
import tensorflow as tf
from tensorflow import keras
print('TensorFlow', tf.__version__)

# ---- figure style: one color and one marker per detector, used in every figure ----
COL = {'IF': '#2a78d6', 'PCA': '#eda100', 'VAE': '#eb6834', 'VAE+RM+MAD': '#1baf7a',
       'Normal': '#9aa0a6', 'Attacked': '#e34948'}
MRK = {'IF': 'o', 'PCA': 'D', 'VAE': 's', 'VAE+RM+MAD': '^'}
LST = {'IF': ':', 'PCA': '-.', 'VAE': '--', 'VAE+RM+MAD': '-'}
plt.rcParams.update({'font.family': 'serif', 'font.size': 10, 'axes.titlesize': 10, 'axes.labelsize': 10,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e6e6', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
                     'legend.frameon': False, 'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

def panel_label(ax, s):
    # (a), (b), ... placed inside the panel, as the report reviewers require
    ax.text(0.02, 0.97, s, transform=ax.transAxes, fontweight='bold', va='top', ha='left',
            bbox=dict(facecolor='white', edgecolor='none', pad=1.5, alpha=0.8))

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + '.png')); plt.show()

def save_table(df, name, index=False):
    df.to_csv(os.path.join(TAB_DIR, name + '.csv'), index=index)
    display(df)

## 1. Data

The raw file has 18,290 records and 87 columns. Fourteen features are used. Twist31 and Curvature were
dropped in the original feature selection for an absolute correlation above 0.85 with other features.

In [ ]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
raw = raw.rename(columns={'BFI L': 'BFI_L_Value', 'BFI C ': 'BFI_C_Value', 'BFI R ': 'BFI_R_Value'})
print('Raw shape:', raw.shape)
df = raw[CONFIG['features']].apply(pd.to_numeric, errors='coerce').dropna().reset_index(drop=True)
print('Used shape:', df.shape, '| missing values:', int(df.isna().sum().sum()))

cols   = list(df.columns)
R      = df.values.astype(float)                 # raw measured values
scaler = StandardScaler().fit(R)
X      = scaler.transform(R)                     # standardized clean data
N      = len(X)

# check of the correlation filter on the 16 candidate features
cand = raw[CONFIG['features'] + ['Twist31', 'Curvature']].apply(pd.to_numeric, errors='coerce').abs().corr().abs()
for c in ['Twist31', 'Curvature']:
    other = cand[c].drop(c).idxmax(); print(f'{c}: highest |r| = {cand[c].drop(c).max():.2f} with {other}')

In [ ]:
# TABLE 1. Input features and descriptive statistics
group = {'LProf62': 'Track geometry', 'RProf62': 'Track geometry', 'LAlign62': 'Track geometry',
         'RAlign62': 'Track geometry', 'Gage': 'Track geometry', 'Warp62': 'Track geometry',
         'Crosslevel': 'Track geometry', 'Left Cant': 'Rail cant', 'Right Cant': 'Rail cant',
         'BFI_L_Value': 'Subsurface condition', 'BFI_C_Value': 'Subsurface condition',
         'BFI_R_Value': 'Subsurface condition', 'TotalLatForce': 'Operating measurements',
         'Speed': 'Operating measurements'}
t1 = df.describe().T[['mean', 'std', 'min', 'max']].round(3)
t1.insert(0, 'Group', [group[c] for c in t1.index]); t1.insert(1, 'Attacked', ['Yes' if c in CONFIG['targets'] else 'No' for c in t1.index])
t1 = t1.reset_index().rename(columns={'index': 'Feature', 'mean': 'Mean', 'std': 'Std. dev.', 'min': 'Min', 'max': 'Max'})
save_table(t1, 'table1_features')

## 2. Detection models

* **Isolation Forest** and **PCA** are fitted on the clean standardized data.
* **VAE**: fully connected encoder (64, 32, 16), latent dimension 7, mirrored decoder. Loss is the sum of squared
  reconstruction error and the KL divergence. Trained on clean data only.
  The anomaly score is the reconstruction error $e = \lVert x - x' \rVert^2$, with $x'$ decoded from the latent mean.

In [ ]:
class Sampling(keras.layers.Layer):
    def call(self, inputs):
        mu, log_var = inputs
        return mu + tf.exp(0.5 * log_var) * keras.random.normal(shape=tf.shape(mu))

class VAE(keras.Model):
    def __init__(self, input_dim, latent_dim, hidden):
        super().__init__()
        self.enc = keras.Sequential([keras.layers.Dense(h, activation='relu') for h in hidden])
        self.mu, self.log_var, self.sample = keras.layers.Dense(latent_dim), keras.layers.Dense(latent_dim), Sampling()
        self.dec = keras.Sequential([keras.layers.Dense(h, activation='relu') for h in hidden[::-1]]
                                    + [keras.layers.Dense(input_dim)])
    def encode(self, x):
        h = self.enc(x); return self.mu(h), self.log_var(h)
    def call(self, x):
        mu, log_var = self.encode(x)
        x_hat = self.dec(self.sample([mu, log_var]))
        recon = tf.reduce_mean(tf.reduce_sum(tf.square(x - x_hat), axis=1))
        kl = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1))
        self.add_loss(recon + kl)
        return x_hat
    def latent(self, x):
        return self.encode(tf.constant(x, dtype=tf.float32))[0].numpy()
    def residual(self, x):                      # x - x', decoded from the latent mean
        return x - self.dec(self.encode(tf.constant(x, dtype=tf.float32))[0]).numpy()

def train_vae(X_train, seed):
    keras.utils.set_random_seed(seed)
    vae = VAE(X_train.shape[1], CONFIG['latent_dim'], CONFIG['hidden'])
    vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    hist = vae.fit(X_train, X_train, epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'],
                   validation_split=0.2, verbose=0,
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=CONFIG['patience'],
                                                            restore_best_weights=True)])
    return vae, hist.history

## 3. Statistical scoring and the MAD threshold

**Hybrid (VAE+RM+MAD).** The VAE error sequence $e_t$ is passed through a centered rolling median of width $w$

$$\tilde e_t = \mathrm{median}(e_{t-(w-1)/2}, \dots, e_{t+(w-1)/2})$$

and a record is flagged when

$$\tilde e_t > \mathrm{median}(\tilde E) + K \cdot 1.4826 \cdot \mathrm{MAD}(\tilde E)$$

where $\tilde E$ is the set of filtered errors on **clean** data. No attack labels are used.
Two values of $K$ are reported: the conventional $K = 3$, and a calibrated $K$ chosen so that 1% of clean
records exceed the threshold.

The same MAD threshold rule is applied to every detector, so that detectors are compared on equal terms.

In [ ]:
def robust_fit(v):
    med = np.median(v, axis=0); mad = 1.4826 * np.median(np.abs(v - med), axis=0) + 1e-12
    return med, mad

def rolling_median(s, w):
    return pd.Series(s).rolling(w, center=True, min_periods=1).median().values

def ewma(s, lam):
    out = np.empty(len(s)); acc = s[0]
    for i, v in enumerate(s):
        acc = lam * v + (1 - lam) * acc; out[i] = acc
    return out

def cusum(z, k):
    out = np.empty(len(z)); c = 0.0
    for i, v in enumerate(z):
        c = max(0.0, c + v - k); out[i] = c
    return out

def mad_threshold(clean_score, k):
    med, mad = robust_fit(clean_score); return med + k * mad

def calibrated_k(clean_score, far):
    # K such that a share `far` of clean records exceed median + K * MAD
    med, mad = robust_fit(clean_score); return (np.quantile(clean_score, 1 - far) - med) / mad

class Detectors:
    # Everything is fitted or calibrated on clean training data only.
    def __init__(self, X_train, seed):
        self.iso = IsolationForest(contamination=0.05, random_state=seed).fit(X_train)
        self.pca = PCA(n_components=CONFIG['latent_dim']).fit(X_train)
        self.vae, self.history = train_vae(X_train, seed)
        r = self.vae.residual(X_train)
        self.rz = robust_fit(r)
        self.mcd = MinCovDet(random_state=0).fit(r)
        self.logfit = robust_fit(np.log((r ** 2).sum(1) + 1e-12))
        self.xfit = robust_fit(X_train)
    def vae_error(self, X):
        return (self.vae.residual(X) ** 2).sum(1)
    def scores(self, X, rm_window=None):
        w = rm_window or CONFIG['rm_window']
        r = self.vae.residual(X); err = (r ** 2).sum(1)
        zx = (((X - self.xfit[0]) / self.xfit[1]) ** 2).sum(1)
        return {
            'IF': -self.iso.score_samples(X),
            'PCA': ((X - self.pca.inverse_transform(self.pca.transform(X))) ** 2).sum(1),
            'VAE': err,
            'VAE+RM+MAD': rolling_median(err, w),
            'VAE+EWMA': ewma(err, CONFIG['ewma_lambda']),
            'VAE+CUSUM': cusum((np.log(err + 1e-12) - self.logfit[0]) / self.logfit[1], CONFIG['cusum_k']),
            'VAE+Mahalanobis': np.sqrt(self.mcd.mahalanobis(r)),
            'VAE+robust z': (((r - self.rz[0]) / self.rz[1]) ** 2).sum(1),
            'Robust z (no VAE)': zx,
            'RM robust z (no VAE)': rolling_median(zx, w),
        }

def metrics(y, yp):
    return dict(precision=precision_score(y, yp, zero_division=0), recall=recall_score(y, yp, zero_division=0),
                f1=f1_score(y, yp, zero_division=0), accuracy=accuracy_score(y, yp))

def best_f1_cut(score, y):
    order = np.argsort(-score); s, yy = score[order], y[order]
    f1 = 2 * np.cumsum(yy) / (np.arange(1, len(s) + 1) + yy.sum()); j = int(np.argmax(f1))
    return (s[j] + s[j + 1]) / 2 if j + 1 < len(s) else s[j] - 1e-12

def cv_predict(score, y, seed):
    # threshold chosen on four folds with the attack labels, applied to the held out fold
    yp = np.zeros_like(y)
    for tr, te in StratifiedKFold(CONFIG['n_folds'], shuffle=True, random_state=seed).split(score, y):
        yp[te] = (score[te] > best_f1_cut(score[tr], y[tr])).astype(int)
    return yp

## 4. False data injection

Attacks are injected into the **raw measured values** of Gage, Warp62 and Crosslevel, in windows of 46 consecutive
records spaced evenly along the track. Each window uses one technique.

| Technique | Raw scenario | Subtle scenario |
|---|---|---|
| Additive spike | value ± (2 to 3) × feature standard deviation | same |
| Multiplicative | value × 1.5 or × 2 | deviation from the feature median × 1.5 or × 2 |
| Random noise | Gaussian noise, sd = 3 × feature standard deviation | same |

The attacked data are standardized with the scaler fitted on clean data.

In [ ]:
def inject(R_clean, n_windows, rng, scenario='raw'):
    R = R_clean.copy(); n = len(R); L = CONFIG['window_len']
    y = np.zeros(n, dtype=int); kind = np.array([''] * n, dtype=object)
    spread = (n - n_windows * L) // n_windows
    idx = np.array([cols.index(c) for c in CONFIG['targets']])
    sd = R_clean[:, idx].std(0); med = np.median(R_clean[:, idx], axis=0)
    for i in range(n_windows):
        s = i * (spread + L); e = s + L
        block = R[s:e, :][:, idx]
        method = str(rng.choice(['spike', 'multiply', 'noise']))
        if method == 'spike':
            block = block + rng.choice([-1, 1]) * rng.uniform(2, 3) * sd
        elif method == 'multiply':
            f = rng.choice([1.5, 2.0])
            block = block * f if scenario == 'raw' else med + (block - med) * f
        else:
            block = block + rng.normal(0, CONFIG['noise_std'], size=block.shape) * sd
        R[s:e, idx] = block; y[s:e] = 1; kind[s:e] = method
    return R, y, kind

_R, _y, _k = inject(R, 20, np.random.default_rng(0))
print('5% level:', _y.sum(), 'attacked records =', round(100 * _y.mean(), 2), '% |', pd.Series(_k[_y == 1]).value_counts().to_dict())

## 5. Run the experiment

For every seed: fit the detectors on clean data, inject attacks (two scenarios, three attack levels), score, and
evaluate with four thresholds.

| Threshold | Uses attack labels? | Rule |
|---|---|---|
| MAD (K = 3) | No | median + 3 × 1.4826 × MAD of clean scores |
| MAD (calibrated) | No | K set so that 1% of clean records are flagged |
| Percentile | No, but assumes a 5% attack share | 95th percentile of the scores being evaluated |
| Cross validation | Yes | cut that maximizes F1 on four folds, applied to the fifth |

In [ ]:
rows, rec_rows, sens_rows, REF = [], [], [], {}
t0 = time.time()
for seed in CONFIG['seeds']:
    det = Detectors(X, seed)
    clean = det.scores(X)
    kcal = {m: calibrated_k(s, CONFIG['target_far']) for m, s in clean.items()}
    for scenario in ['raw', 'subtle']:
        for level, n_win in CONFIG['windows'].items():
            Ri, y, kind = inject(R, n_win, np.random.default_rng(1000 * seed + n_win), scenario)
            Xi = scaler.transform(Ri); sc = det.scores(Xi)
            for m, s in sc.items():
                pred = {'MAD (K = 3)': (s > mad_threshold(clean[m], CONFIG['k_mad'])).astype(int),
                        'MAD (calibrated)': (s > mad_threshold(clean[m], kcal[m])).astype(int),
                        'Percentile': (s > np.percentile(s, CONFIG['percentile'])).astype(int),
                        'Cross validation': cv_predict(s, y, seed)}
                auc, ap = roc_auc_score(y, s), average_precision_score(y, s)
                for th, yp in pred.items():
                    rows.append(dict(seed=seed, scenario=scenario, level=f'{int(level*100)}%', model=m, threshold=th,
                                     auc=auc, ap=ap, k_cal=kcal[m], far=1 - recall_score(1 - y, 1 - yp), **metrics(y, yp)))
                    for a in ['spike', 'multiply', 'noise']:
                        if (kind == a).any():
                            rec_rows.append(dict(seed=seed, scenario=scenario, level=f'{int(level*100)}%', model=m,
                                                 threshold=th, attack=a, recall=yp[kind == a].mean()))
            if level == 0.05 and scenario == 'raw':
                # sensitivity of the hybrid to the rolling window and to the MAD multiplier
                err, err_c = sc['VAE'], clean['VAE']
                for w in CONFIG['rm_window_grid']:
                    s, c = rolling_median(err, w), rolling_median(err_c, w)
                    sens_rows.append(dict(seed=seed, kind='window', value=w, auc=roc_auc_score(y, s),
                                          f1_cv=f1_score(y, cv_predict(s, y, seed)),
                                          f1_mad_cal=f1_score(y, (s > mad_threshold(c, calibrated_k(c, CONFIG['target_far']))).astype(int))))
                s, c = sc['VAE+RM+MAD'], clean['VAE+RM+MAD']
                for k in CONFIG['k_grid'] + [kcal['VAE+RM+MAD']]:
                    yp = (s > mad_threshold(c, k)).astype(int)
                    sens_rows.append(dict(seed=seed, kind='k', value=round(float(k), 2), calibrated=(k == kcal['VAE+RM+MAD']),
                                          far=(c > mad_threshold(c, k)).mean(), **metrics(y, yp)))
                if seed == CONFIG['seeds'][0]:      # kept for the figures
                    REF = dict(det=det, Xi=Xi, y=y, kind=kind, scores=sc, clean=clean, kcal=kcal)
    print(f'seed {seed} done ({time.time() - t0:.0f} s)')

res  = pd.DataFrame(rows);     res.to_csv(os.path.join(TAB_DIR, 'all_runs.csv'), index=False)
rec  = pd.DataFrame(rec_rows); rec.to_csv(os.path.join(TAB_DIR, 'all_recall_by_attack.csv'), index=False)
sens = pd.DataFrame(sens_rows)
print('rows:', len(res))

## 6. Tables

In [ ]:
def pm(g, c, d=3):                       # mean ± standard deviation across seeds
    return f'{g[c].mean():.{d}f} ± {g[c].std():.{d}f}'

def result_table(sub, by, metrics_=('precision', 'recall', 'f1', 'accuracy')):
    out = []
    for key, g in sub.groupby(by, sort=False):
        key = key if isinstance(key, tuple) else (key,)
        out.append({**dict(zip(by, key)), **{c.capitalize() if c != 'f1' else 'F1': pm(g, c) for c in metrics_},
                    'AUC': pm(g, 'auc')})
    return pd.DataFrame(out)

main = res[res.model.isin(MAIN_MODELS)]
order = lambda d: d.assign(_m=d.model.map({m: i for i, m in enumerate(MAIN_MODELS + ABLATION)})).sort_values(
    [c for c in ['scenario', 'level_n', 'threshold', '_m'] if c in d.columns or c == '_m']).drop(columns='_m')

# TABLE 2. Main result at the 5% attack level (raw scenario), all thresholds
t2 = result_table(main[(main.scenario == 'raw') & (main.level == '5%')], ['threshold', 'model'])
save_table(t2.rename(columns={'threshold': 'Threshold', 'model': 'Model'}), 'table2_main_5pct')

In [ ]:
# TABLE 3. Performance across attack levels (raw scenario)
t3 = []
for th in ['MAD (calibrated)', 'Cross validation']:
    t = result_table(main[(main.scenario == 'raw') & (main.threshold == th)], ['level', 'model']); t.insert(0, 'Threshold', th); t3.append(t)
t3 = pd.concat(t3).rename(columns={'level': 'Attack level', 'model': 'Model'})
save_table(t3, 'table3_attack_levels')

In [ ]:
# TABLE 4. Paired comparison of the hybrid against the standalone VAE (same seed, same attacks)
out = []
for (scn, lvl, th), g in res[res.model.isin(['VAE', 'VAE+RM+MAD'])].groupby(['scenario', 'level', 'threshold'], sort=False):
    p = g.pivot(index='seed', columns='model', values='f1'); d = p['VAE+RM+MAD'] - p['VAE']
    pval = stats.wilcoxon(d).pvalue if (d != 0).any() else 1.0
    out.append({'Scenario': scn, 'Attack level': lvl, 'Threshold': th, 'VAE F1': round(p['VAE'].mean(), 3),
                'Hybrid F1': round(p['VAE+RM+MAD'].mean(), 3), 'Mean difference': round(d.mean(), 3),
                'Seeds won': f'{int((d > 0).sum())} of {len(d)}', 'Wilcoxon p': round(pval, 4)})
t4 = pd.DataFrame(out)
save_table(t4, 'table4_hybrid_vs_vae')

In [ ]:
# TABLE 5. Ablation. Other statistical models after the VAE, and detectors without a VAE (5% level)
out = []
for scn in ['raw', 'subtle']:
    g0 = res[(res.scenario == scn) & (res.level == '5%') & res.model.isin(ABLATION)]
    for m in ABLATION:
        g = g0[g0.model == m]; cv = g[g.threshold == 'Cross validation']; mc = g[g.threshold == 'MAD (calibrated)']
        base = g0[(g0.model == 'VAE') & (g0.threshold == 'Cross validation')].set_index('seed').f1
        d = cv.set_index('seed').f1 - base
        out.append({'Scenario': scn, 'Model': m, 'AUC': round(cv.auc.mean(), 3), 'Average precision': round(cv.ap.mean(), 3),
                    'F1 (cross validation)': pm(cv, 'f1'), 'F1 (MAD calibrated)': pm(mc, 'f1'),
                    'Seeds won against VAE': '' if m == 'VAE' else f'{int((d > 0).sum())} of {len(d)}'})
t5 = pd.DataFrame(out)
save_table(t5, 'table5_ablation_statistical_models')

In [ ]:
# TABLE 6. Sensitivity of the hybrid (5% level, raw scenario)
w = sens[sens.kind == 'window'].groupby('value')[['auc', 'f1_cv', 'f1_mad_cal']].agg(['mean', 'std']).round(3)
w.columns = [f'{a} {b}' for a, b in w.columns]; w = w.reset_index().rename(columns={'value': 'Rolling window (records)'})
save_table(w, 'table6a_window_sensitivity')

k = sens[sens.kind == 'k'].copy(); k['K'] = np.where(k.calibrated == True, 'calibrated', k.value.astype(str))
kt = k.groupby('K', sort=False)[['value', 'far', 'precision', 'recall', 'f1']].mean().round(3).reset_index()
kt = kt.rename(columns={'value': 'Mean K', 'far': 'Clean false alarm rate', 'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1'})
save_table(kt, 'table6b_mad_multiplier_sensitivity')

In [ ]:
# TABLE 7. Recall by attack technique (all attack levels pooled, MAD calibrated threshold)
t7 = (rec[rec.model.isin(MAIN_MODELS) & (rec.threshold == 'MAD (calibrated)')]
      .groupby(['scenario', 'model', 'attack']).recall.mean().round(3).unstack('attack')[['spike', 'noise', 'multiply']]
      .reindex(MAIN_MODELS, level='model').reset_index()
      .rename(columns={'scenario': 'Scenario', 'model': 'Model', 'spike': 'Additive spike', 'noise': 'Random noise', 'multiply': 'Multiplicative'}))
save_table(t7, 'table7_recall_by_attack')

# TABLE 8. Subtle scenario across attack levels
t8 = []
for th in ['MAD (calibrated)', 'Cross validation']:
    t = result_table(main[(main.scenario == 'subtle') & (main.threshold == th)], ['level', 'model']); t.insert(0, 'Threshold', th); t8.append(t)
t8 = pd.concat(t8).rename(columns={'level': 'Attack level', 'model': 'Model'})
save_table(t8, 'table8_subtle_scenario')

## 7. Figures

All figures use one fixed color, marker and line style per detector. Panels carry (a), (b), ... labels inside the plot.
Figures 2 to 5 use the first seed at the 5% attack level (raw scenario). Figures 6 to 10 use all seeds.

In [ ]:
# FIGURE 1. VAE training curve (first seed)
h = REF['det'].history
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(h['loss'], color=COL['VAE'], lw=1.8, label='Training'); ax.plot(h['val_loss'], color=COL['IF'], lw=1.8, ls='--', label='Validation')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (reconstruction + KL)'); ax.legend()
save_fig(fig, 'fig01_vae_training')

In [ ]:
# FIGURE 2. Anomaly score along the track with the attack windows (5% level)
y, sc, clean, kcal = REF['y'], REF['scores'], REF['clean'], REF['kcal']
names = {'IF': 'Isolation Forest score', 'VAE': 'VAE reconstruction error', 'VAE+RM+MAD': 'Hybrid (VAE+RM+MAD) score'}
fig, axes = plt.subplots(3, 1, figsize=(4.8, 3.9), sharex=True)   # sized for print at about 4 in wide
edges = np.flatnonzero(np.diff(np.r_[0, y, 0]))
for ax, m, lab in zip(axes, names, 'abc'):
    for s, e in zip(edges[::2], edges[1::2]):
        ax.axvspan(s, e, color=COL['Attacked'], alpha=0.18, lw=0)
    ax.plot(sc[m], color=COL[m], lw=0.5)
    ax.axhline(mad_threshold(clean[m], kcal[m]), color='k', ls='--', lw=1)
    if m != 'IF': ax.set_yscale('log')
    ax.set_ylabel({'IF': 'IF score', 'VAE': 'VAE error', 'VAE+RM+MAD': 'Hybrid score'}[m]); panel_label(ax, f'({lab})')
axes[0].plot([], [], color='k', ls='--', lw=1, label='MAD threshold (calibrated)'); axes[0].fill_between([], [], color=COL['Attacked'], alpha=0.18, label='Attack window')
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper center', ncol=2, bbox_to_anchor=(0.55, 1.03), fontsize=8)
axes[-1].set_xlabel('Record index along the track'); axes[-1].set_xticks([0, 5000, 10000, 15000]); fig.tight_layout(rect=(0, 0, 1, 0.96))
save_fig(fig, 'fig02_scores_along_track')

In [ ]:
# FIGURE 3. Score distributions of normal and attacked records (5% level)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, m, lab in zip(axes, names, 'abc'):
    s = sc[m]; lo, hi = np.percentile(s, [0.5, 99.9])
    bins = np.linspace(lo, hi, 50) if m == 'IF' else np.logspace(np.log10(max(lo, 1e-3)), np.log10(hi), 50)
    ax.hist(s[y == 0], bins=bins, color=COL['Normal'], alpha=0.85, density=True, label='Normal')
    ax.hist(s[y == 1], bins=bins, color=COL['Attacked'], alpha=0.65, density=True, label='Attacked')
    ax.axvline(mad_threshold(clean[m], kcal[m]), color='k', ls='--', lw=1, label='MAD threshold')
    if m != 'IF': ax.set_xscale('log')
    ax.set_xlabel(names[m]); ax.set_ylabel('Density' if lab == 'a' else ''); panel_label(ax, f'({lab})')
axes[2].legend(loc='upper right', fontsize=8)
fig.tight_layout(); save_fig(fig, 'fig03_score_distributions')

In [ ]:
# FIGURE 4. PCA and t-SNE views of the attacked dataset (5% level): input features and VAE latent space
rng = np.random.default_rng(0); Xi = REF['Xi']
keep = np.r_[rng.choice(np.flatnonzero(y == 0), 3000, replace=False), np.flatnonzero(y == 1)]
Z = REF['det'].vae.latent(Xi)
views = [('PCA of input features', PCA(2).fit_transform(Xi)[keep]), ('PCA of VAE latent space', PCA(2).fit_transform(Z)[keep]),
         ('t-SNE of input features', TSNE(2, random_state=0, init='pca').fit_transform(Xi[keep])),
         ('t-SNE of VAE latent space', TSNE(2, random_state=0, init='pca').fit_transform(Z[keep]))]
fig, axes = plt.subplots(2, 2, figsize=(4.8, 4.4))   # sized for print at about 3.6 in wide
for ax, (title, P), lab in zip(axes.ravel(), views, 'abcd'):
    yk = y[keep]
    ax.scatter(P[yk == 0, 0], P[yk == 0, 1], s=2, color=COL['Normal'], alpha=0.5, label='Normal', rasterized=True)
    ax.scatter(P[yk == 1, 0], P[yk == 1, 1], s=4, color=COL['Attacked'], alpha=0.7, marker='x', lw=0.6, label='Attacked', rasterized=True)
    ax.set_title(title, fontsize=8.5); ax.set_xlabel('Component 1', fontsize=8.5); ax.set_ylabel('Component 2', fontsize=8.5); ax.tick_params(labelsize=8); panel_label(ax, f'({lab})')
axes[0, 1].legend(loc='lower left', markerscale=3, fontsize=7.5)
fig.tight_layout(); save_fig(fig, 'fig04_pca_tsne')

In [ ]:
# FIGURE 5. ROC curves at the 5% attack level (first seed)
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for m in MAIN_MODELS:
    fpr, tpr, _ = roc_curve(y, sc[m]); ax.plot(fpr, tpr, color=COL[m], ls=LST[m], lw=1.8, label=f'{m} (AUC = {roc_auc_score(y, sc[m]):.3f})')
ax.plot([0, 1], [0, 1], color='#bbbbbb', lw=0.8); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(loc='lower right')
save_fig(fig, 'fig05_roc')

In [ ]:
# FIGURE 6. Precision, recall and F1 at the 5% attack level under each threshold (mean of all seeds, bars show standard deviation)
ths = ['MAD (K = 3)', 'MAD (calibrated)', 'Percentile', 'Cross validation']
g = main[(main.scenario == 'raw') & (main.level == '5%')]
fig, axes = plt.subplots(1, 4, figsize=(11, 3.2), sharey=True)
xpos = np.arange(3); wbar = 0.2
for ax, th, lab in zip(axes, ths, 'abcd'):
    for i, m in enumerate(MAIN_MODELS):
        s = g[(g.threshold == th) & (g.model == m)][['precision', 'recall', 'f1']]
        ax.bar(xpos + (i - 1.5) * wbar, s.mean(), wbar * 0.9, yerr=s.std(), color=COL[m], label=m, error_kw=dict(lw=0.8, capsize=2))
    ax.set_xticks(xpos); ax.set_xticklabels(['Precision', 'Recall', 'F1']); ax.set_ylim(0, 1.05); ax.set_title(th); panel_label(ax, f'({lab})')
axes[0].set_ylabel('Score'); fig.tight_layout()
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.09))
save_fig(fig, 'fig06_thresholds_5pct')

In [ ]:
# FIGURE 7. Performance across attack levels (cross validation threshold, mean of all seeds, bars show standard deviation)
lv = ['2%', '5%', '10%']
def level_figure(scenario, threshold, name):
    g = main[(main.scenario == scenario) & (main.threshold == threshold)]
    fig, axes = plt.subplots(1, 4, figsize=(11, 3), sharex=True)
    for ax, met, lab in zip(axes, ['precision', 'recall', 'f1', 'accuracy'], 'abcd'):
        for m in MAIN_MODELS:
            s = g[g.model == m].groupby('level')[met].agg(['mean', 'std']).reindex(lv)
            ax.errorbar(lv, s['mean'], yerr=s['std'], color=COL[m], marker=MRK[m], ls=LST[m], lw=1.6, ms=5, capsize=2, label=m)
        ax.set_title('F1 score' if met == 'f1' else met.capitalize()); ax.set_xlabel('Attack level'); ax.set_ylim(0, 1.05); panel_label(ax, f'({lab})')
    axes[0].set_ylabel('Score'); fig.tight_layout()
    fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.1))
    save_fig(fig, name)
level_figure('raw', 'Cross validation', 'fig07_attack_levels_cv')

In [ ]:
# FIGURE 8. Same as Figure 7 with the label free MAD (calibrated) threshold
level_figure('raw', 'MAD (calibrated)', 'fig08_attack_levels_mad')

In [ ]:
# FIGURE 9. Recall by attack technique, (a) raw scenario and (b) subtle scenario (MAD calibrated threshold)
att = ['spike', 'noise', 'multiply']; attn = ['Additive spike', 'Random noise', 'Multiplicative']
g = rec[rec.model.isin(MAIN_MODELS) & (rec.threshold == 'MAD (calibrated)')]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for ax, scn, lab in zip(axes, ['raw', 'subtle'], 'ab'):
    for i, m in enumerate(MAIN_MODELS):
        s = g[(g.scenario == scn) & (g.model == m)].groupby('attack').recall.mean().reindex(att)
        ax.bar(np.arange(3) + (i - 1.5) * 0.2, s, 0.18, color=COL[m], label=m)
    ax.set_xticks(np.arange(3)); ax.set_xticklabels(attn); ax.set_ylim(0, 1.05); ax.set_title(f'{scn.capitalize()} scenario'); panel_label(ax, f'({lab})')
axes[0].set_ylabel('Recall'); axes[1].legend(loc='upper right', fontsize=7.5)
fig.tight_layout(); save_fig(fig, 'fig09_recall_by_attack')

In [ ]:
# FIGURE 10. Ablation of the statistical model at the 5% attack level, (a) raw and (b) subtle scenario
g = res[(res.level == '5%') & (res.threshold == 'Cross validation') & res.model.isin(ABLATION)]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, scn, lab in zip(axes, ['raw', 'subtle'], 'ab'):
    s = g[g.scenario == scn].groupby('model').f1.agg(['mean', 'std']).reindex(ABLATION[::-1])
    colors = [COL.get(m, '#b9bec4') for m in s.index]
    ax.barh(s.index, s['mean'], xerr=s['std'], color=colors, error_kw=dict(lw=0.8, capsize=2))
    ax.axvline(s.loc['VAE', 'mean'], color=COL['VAE'], ls='--', lw=1)
    ax.set_xlim(0, 1); ax.set_xlabel('F1 score (cross validation threshold)'); ax.set_title(f'{scn.capitalize()} scenario')
    ax.text(0.97, 0.05, f'({lab})', transform=ax.transAxes, fontweight='bold', ha='right')
fig.tight_layout(); save_fig(fig, 'fig10_ablation')

In [ ]:
# FIGURE 11. Sensitivity of the hybrid, (a) rolling window and (b) MAD multiplier (5% level, raw scenario)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.1))
s = sens[sens.kind == 'window'].groupby('value')[['f1_cv', 'f1_mad_cal']].agg(['mean', 'std'])
vae_cv = res[(res.model == 'VAE') & (res.scenario == 'raw') & (res.level == '5%') & (res.threshold == 'Cross validation')].f1.mean()
axes[0].errorbar(s.index, s[('f1_cv', 'mean')], yerr=s[('f1_cv', 'std')], color=COL['VAE+RM+MAD'], marker='^', lw=1.6, capsize=2, label='Hybrid, cross validation')
axes[0].errorbar(s.index, s[('f1_mad_cal', 'mean')], yerr=s[('f1_mad_cal', 'std')], color=COL['VAE+RM+MAD'], marker='v', ls='--', lw=1.6, capsize=2, label='Hybrid, MAD (calibrated)')
axes[0].axhline(vae_cv, color=COL['VAE'], ls=':', lw=1.4, label='VAE, cross validation')
axes[0].set_xlabel('Rolling median window (records)'); axes[0].set_ylabel('F1 score'); axes[0].set_ylim(0, 1.05); axes[0].set_xticks(list(s.index)); axes[0].legend(fontsize=7.5, loc='lower right'); panel_label(axes[0], '(a)')
k = sens[(sens.kind == 'k') & (sens.calibrated != True)].groupby('value')[['precision', 'recall', 'f1']].mean()
for c, mk, ls in [('precision', 'o', ':'), ('recall', 's', '--'), ('f1', '^', '-')]:
    axes[1].plot(k.index, k[c], marker=mk, ls=ls, lw=1.6, color={'precision': COL['IF'], 'recall': COL['VAE'], 'f1': COL['VAE+RM+MAD']}[c], label='F1' if c == 'f1' else c.capitalize())
axes[1].set_xlabel('MAD multiplier K'); axes[1].set_ylim(0, 1.05); axes[1].set_xticks(list(k.index)); axes[1].legend(fontsize=7.5, loc='lower right'); panel_label(axes[1], '(b)')
fig.tight_layout(); save_fig(fig, 'fig11_sensitivity')

## 8. Robustness check: records the detectors have never seen

In Sections 5 to 7 the detectors are fitted on the clean version of the same records that are later attacked, as in
the original study. Two stricter tests are run here (5% level, raw scenario). In both, the detectors are fitted and
calibrated on training records only, and attacks are injected into test records the detectors have never seen.

| Split | Training records | Test records |
|---|---|---|
| Interleaved blocks | every other block of 500 consecutive records | the remaining blocks |
| Contiguous | the first 60% of the track | the last 40% of the track |

The interleaved split tests unseen records from the same parts of the track. The contiguous split also tests whether
the detectors transfer to a part of the track they were not trained on.

In [ ]:
BLOCK = 500
blk = (np.arange(N) // BLOCK) % 2
splits = {'Interleaved blocks': (np.flatnonzero(blk == 0), np.flatnonzero(blk == 1)),
          'Contiguous': (np.arange(int(CONFIG['holdout_train_frac'] * N)), np.arange(int(CONFIG['holdout_train_frac'] * N), N))}
hold = []
for split, (tr, te) in splits.items():
    sc_h = StandardScaler().fit(R[tr]); Xtr, Rte = sc_h.transform(R[tr]), R[te]
    n_win = int(round(0.05 * len(Rte) / CONFIG['window_len']))
    for seed in CONFIG['holdout_seeds']:
        det = Detectors(Xtr, seed); clean = det.scores(Xtr)
        Ri, yh, _ = inject(Rte, n_win, np.random.default_rng(7000 + seed)); s_all = det.scores(sc_h.transform(Ri))
        clean_te = det.scores(sc_h.transform(Rte))
        for m in MAIN_MODELS:
            s = s_all[m]; thr = mad_threshold(clean[m], calibrated_k(clean[m], CONFIG['target_far']))
            mm, mc = metrics(yh, (s > thr).astype(int)), metrics(yh, cv_predict(s, yh, seed))
            hold.append({'Split': split, 'seed': seed, 'Model': m, 'AUC': roc_auc_score(yh, s),
                         'Precision (MAD calibrated)': mm['precision'], 'Recall (MAD calibrated)': mm['recall'],
                         'F1 (MAD calibrated)': mm['f1'], 'F1 (cross validation)': mc['f1'],
                         'False alarm rate on unseen clean records': (clean_te[m] > thr).mean()})
        print(split, 'seed', seed, 'done')
    print(f'{split}: training records {len(tr)}, test records {len(te)}, attacked {int(yh.sum())}')
hold = pd.DataFrame(hold); hold.to_csv(os.path.join(TAB_DIR, 'holdout_all_runs.csv'), index=False)
t9 = hold.groupby(['Split', 'Model'], sort=False).mean(numeric_only=True).drop(columns='seed').round(3).reset_index()
save_table(t9, 'table9_holdout')

## 9. Outputs

| File | Content |
|---|---|
| `tables/table1_features.csv` | Input features and descriptive statistics |
| `tables/table2_main_5pct.csv` | Main result at the 5% attack level, all thresholds |
| `tables/table3_attack_levels.csv` | Performance at 2%, 5% and 10% |
| `tables/table4_hybrid_vs_vae.csv` | Paired comparison of the hybrid against the VAE |
| `tables/table5_ablation_statistical_models.csv` | Other statistical models, and detectors without a VAE |
| `tables/table6a_window_sensitivity.csv`, `table6b_mad_multiplier_sensitivity.csv` | Sensitivity of the hybrid |
| `tables/table7_recall_by_attack.csv` | Recall by attack technique |
| `tables/table8_subtle_scenario.csv` | Subtle scenario |
| `tables/table9_holdout.csv` | Records the detectors have never seen |
| `tables/all_runs.csv`, `all_recall_by_attack.csv`, `holdout_all_runs.csv` | Every run, one row per seed |
| `figures/fig01` to `fig11` | Figures at 300 dpi |
| `config.json` | All settings of this run |

In [ ]:
print('Tables :', sorted(os.listdir(TAB_DIR)))
print('Figures:', sorted(os.listdir(FIG_DIR)))